# Kolokvijum II — teorijska osnova

Ovaj notebook **ne rešava zadatak**. Objašnjava koncepte na kojima Kolokvijum II stoji, svaki na
generičkom primeru i uz sliku iz stvarnog sveta. Nigde se ne pominju `Task` ni `Workflow` —
kad ti koncepti sednu, zadatak je samo njihova primena.

| Odeljak | Koncept | Slika |
| --- | --- | --- |
| 1 | funkcija kao podatak | recept naspram kuvanja |
| 2 | redjuser | kasirka na kasi |
| 3 | **transdjuser** | traka sa kutijama i radnici pored nje |
| 4 | kompozicija | raspored radnika duž trake |
| 5 | asinhrono i `Promise` | priznanica u pekari |
| 6 | sekvencijalno izvršavanje | šalteri u opštini |
| 7 | iterabilni protokol | redomat u banci |
| 8 | `map`/`filter`/`flat`/`flatMap` | paketi u pošti |
| 9 | dekorisanje | sekretarica ispred direktora |
| 10 | nepromenljivost | spisak zakucan na tabli |

Odeljak 3 je srce svega ostalog i namerno je najduži.

---
# 1 · Funkcija kao podatak

## Slika: recept naspram kuvanja

**Recept** je list papira. Na njemu piše šta treba uraditi, ali sam po sebi ne radi ništa —
možeš ga presaviti, staviti u fioku, poslati poštom, dati kuvaru, umnožiti u sto primeraka.
Ništa od toga ne pravi jelo.

**Kuvanje** je kad neko uzme taj recept i **izvede** ga. Tek tada nastaje jelo.

U JavaScript-u je isto: `f` je recept, `f()` je kuvanje. Zagrade su razlika između predmeta i radnje.

Zato funkcija može da bude **podatak**: da stoji u polju objekta, u nizu, da se prosledi kao argument
ili vrati kao rezultat. Program koji to koristi ne mora unapred da zna šta se kuva — dobija recept
u trenutku poziva.

In [ ]:
// recept je predmet: možeš ga čuvati i deliti, a da se ništa ne izvrši
const receptZaPozdrav = ime => `Zdravo, ${ime}!`;

const fioka = { naziv: "pozdrav", recept: receptZaPozdrav };   // recept u fioci
const knjigaRecepata = [receptZaPozdrav, x => x.toUpperCase()]; // recepti u nizu

console.log("recept kao vrednost:", typeof receptZaPozdrav);
console.log("nista se nije desilo:", fioka.recept === receptZaPozdrav);

console.log("\ntek sad se kuva:", fioka.recept("Ana"));
console.log("i ovde:          ", knjigaRecepata[0]("Marko"));

// funkcija koja PRIMA recept — ne zna unapred šta radi
const posluziTrojici = recept => ["Ana", "Marko", "Iva"].map(recept);
console.log("\nisti kuvar, prvi recept: ", posluziTrojici(receptZaPozdrav));
console.log("isti kuvar, drugi recept:", posluziTrojici(x => x.length));

// funkcija koja VRAĆA recept — fabrika recepata
const receptZa = pozdrav => ime => `${pozdrav}, ${ime}!`;
const dobroJutro = receptZa("Dobro jutro");
console.log("\nnapravljen recept:", dobroJutro("Ana"));

### Zašto je to važno

Bez ovoga bi svaka nova varijanta posla tražila novu granu u kodu (`if (vrsta === "pozdrav") …`).
Sa ovim, posao se **prosleđuje spolja**, pa **izvršilac** — funkcija koja recept primenjuje, ovde
`posluziTrojici` — ostaje nepromenjen. Menja se samo **poziv**, jer on bira recept.
Ta podela na *ko obilazi* i *šta se radi* je osobina na kojoj počiva sve ostalo u ovom notebooku;
razrađena je u P1 na kraju.

---
# 2 · Redjuser

## Slika: kasirka na kasi

Kroz kasu prolaze artikli, jedan po jedan. Kasirka pored sebe ima **jednu stvar koja se menja** —
zbir na displeju. Za svaki artikl radi isti potez: *uzme dosadašnji zbir i novi artikl, i napravi
novi zbir*.

To je sve što redjuser jeste:

```
(dosadašnje stanje, nova stavka) → novo stanje
```

Ključno: **ista traka artikala, različite kasirke, različit ishod.** Jedna sabira cene. Druga broji
komade. Treća ih pakuje u kesu. Artikli su isti; razlikuje se samo šta kasirka radi sa njima.

`reduce` je „pusti sve artikle kroz ovu kasirku i reci mi šta je na kraju ostalo".

In [ ]:
const artikli = [
    { naziv: "hleb",   cena:  90, kolicina: 2 },
    { naziv: "mleko",  cena: 130, kolicina: 1 },
    { naziv: "kafa",   cena: 850, kolicina: 1 },
    { naziv: "jabuke", cena: 200, kolicina: 3 }
];

// tri kasirke — isti oblik, različit posao
const saberiCene   = (zbir, a)  => zbir + a.cena * a.kolicina;
const prebrojKomade = (broj, a) => broj + a.kolicina;
const uKesu        = (kesa, a)  => [...kesa, a.naziv];
const najskupljiOd = (najskuplji, a) => a.cena > najskuplji.cena ? a : najskuplji;

console.log("kasirka 1 — racun:  ", artikli.reduce(saberiCene, 0));
console.log("kasirka 2 — komada: ", artikli.reduce(prebrojKomade, 0));
console.log("kasirka 3 — kesa:   ", artikli.reduce(uKesu, []));
console.log("kasirka 4 — najskuplji:", artikli.reduce(najskupljiOd, artikli[0]).naziv);

// akumulator ne mora biti broj ni niz — može biti bilo šta
const poCeni = (police, a) => ({ ...police, [a.cena > 200 ? "skupo" : "jeftino"]:
                                 [...(police[a.cena > 200 ? "skupo" : "jeftino"] ?? []), a.naziv] });
console.log("\nkasirka 5 — dve police:", artikli.reduce(poCeni, {}));

### Šta treba zapamtiti

**Redjuser je jedini oblik koji `reduce` razume.** Šta god da hoćeš da dobiješ — broj, niz, objekat,
tekst — moraš to izraziti kao „kako se dosadašnje stanje i jedna nova stavka spajaju u novo stanje".

Zbog toga je redjuser **mesto gde se odlučuje šta je ishod**. Zapamti tu ulogu — u sledećem odeljku
ćemo je odvojiti od pitanja *šta se usput radi sa stavkama*, i tu nastaje transdjuser.

---
# 3 · Transdjuser

## Slika: traka sa kutijama

Zamisli pogon. Kroz njega ide **jedna pokretna traka**, a po njoj putuju kutije.

- Na **kraju trake** stoji **pakerica**. Ona odlučuje šta se dešava sa kutijom koja stigne do nje:
  slaže ih u gajbu, ili samo broji, ili meri ukupnu težinu. To je **redjuser** iz prethodnog odeljka.
- **Pored trake** stoje **radnici**. Svaki radi tačno jednu stvar sa kutijom koja naiđe, pa je
  **doda sledećem u nizu**. Jedan farba, drugi baca oštećene, treći veliku kutiju rasparča na dve.

Dva pravila po kojima pogon radi:

1. **Radnik ne zna ko je posle njega.** Njemu se *kaže* kome da doda kutiju. Zato radnik nije samo
   „posao" — on je **posao plus veza ka sledećem**.
2. **Radnik ne zna šta se dešava na kraju trake.** Svejedno mu je da li pakerica slaže u gajbu ili broji.

Iz ta dva pravila sledi sve: isti radnik radi u bilo kom pogonu, u bilo kom rasporedu, sa bilo kojom
pakericom na kraju.

## Zašto uopšte traka — šta je alternativa

Bez trake, svaki posao je **zasebna linija sa gomilom između**: prva linija pregleda sve kutije i
ostavi gomilu ispravnih; druga linija uzme tu gomilu, ofarba sve i ostavi novu gomilu; treća linija…

To je `niz.filter(p).map(f)`. Radi, ali:

- kutije se **prenose i odlažu** između linija — te međugomile su međunizovi u memoriji,
- svaka kutija se **diže i spušta** onoliko puta koliko ima linija.

Traka sve to radi u **jednom prolazu**: kutija uđe, prođe pored svih radnika redom, i izađe na kraju.
Nikakvih gomila između.

In [ ]:
// ── pakerice (krajnji redjuseri) — šta se dešava na KRAJU trake ──────
const uGajbu = (gajba, kutija) => [...gajba, kutija];        // slaže u gajbu
const izmeri = (ukupno, kutija) => ukupno + kutija.tezina;   // meri težinu
const prebroj = (broj, _)       => broj + 1;                 // samo broji

// ── radnici (transdjuseri) — svaki dobija SLEDEĆEG i vraća sebe-povezanog ──
const preslikaj = posao   => sledeci => (stanje, kutija) => sledeci(stanje, posao(kutija));
const izdvoj    = provera => sledeci => (stanje, kutija) => provera(kutija) ? sledeci(stanje, kutija) : stanje;

// pročitaj `preslikaj` naglas:
//   "daj mi posao. daj mi sledećeg radnika.
//    vraćam radnika koji uzme kutiju, uradi posao, i preda je sledećem."

const kutije = [
    { oznaka: "A", tezina: 12, ostecena: false },
    { oznaka: "B", tezina:  4, ostecena: true  },
    { oznaka: "C", tezina: 30, ostecena: false },
    { oznaka: "D", tezina:  7, ostecena: false }
];

const ispravna = k => !k.ostecena;
const ofarbaj  = k => ({ ...k, boja: "plava" });

// jedan radnik + pakerica
console.log("samo izdvajanje:", kutije.reduce(izdvoj(ispravna)(uGajbu), []).map(k => k.oznaka));
console.log("samo farbanje:  ", kutije.reduce(preslikaj(ofarbaj)(uGajbu), []).map(k => k.boja));

// ISTI radnik, DRUGA pakerica na kraju — radnik se ne menja
console.log("\nisti radnik, gajba: ", kutije.reduce(izdvoj(ispravna)(uGajbu), []).length);
console.log("isti radnik, vaga:  ", kutije.reduce(izdvoj(ispravna)(izmeri), 0));
console.log("isti radnik, brojac:", kutije.reduce(izdvoj(ispravna)(prebroj), 0));

## Dva radnika: raspored duž trake

Da bi dva radnika stajala jedan iza drugog, prvom treba **reći da drugom dodaje kutije**.
A drugom treba reći da dodaje **pakerici**. To je sve što spajanje jeste:

```js
izdvoj(ispravna)( preslikaj(ofarbaj)( uGajbu ) )
//     prvi radnik   drugi radnik      pakerica
```

Čita se **spolja ka unutra**: prvi na traci je onaj koji je najviše spolja.

Pošto se ovo ružno gnezdi kad radnika ima pet, postoji funkcija koja ih poređa umesto nas —
o njoj u odeljku 4.

In [ ]:
// dva radnika, jedan iza drugog
const traka = izdvoj(ispravna)(preslikaj(ofarbaj)(uGajbu));

console.log("kroz oba radnika:", kutije.reduce(traka, []).map(k => `${k.oznaka}:${k.boja}`));

// ── DOKAZ da je prolaz jedan ─────────────────────────────────────────
// isti posao napisan na dva načina, uz beleženje svakog poteza

const dnevnikDveLinije = [];
kutije
    .filter(k => { dnevnikDveLinije.push(`pregled ${k.oznaka}`); return ispravna(k); })
    .map(k => { dnevnikDveLinije.push(`farbanje ${k.oznaka}`); return ofarbaj(k); });

const dnevnikJednaTraka = [];
const izdvojSaTragom = sledeci => (stanje, k) => {
    dnevnikJednaTraka.push(`pregled ${k.oznaka}`);
    return ispravna(k) ? sledeci(stanje, k) : stanje;
};
const ofarbajSaTragom = sledeci => (stanje, k) => {
    dnevnikJednaTraka.push(`farbanje ${k.oznaka}`);
    return sledeci(stanje, ofarbaj(k));
};
kutije.reduce(izdvojSaTragom(ofarbajSaTragom(uGajbu)), []);

console.log("\ndve linije sa gomilom između:");
console.log("  ", dnevnikDveLinije.join(" → "));
console.log("jedna traka:");
console.log("  ", dnevnikJednaTraka.join(" → "));

### Pročitaj ta dva reda pažljivo — u njima je cela poenta

**Dve linije:** `pregled A → pregled B → pregled C → pregled D → farbanje A → farbanje C → farbanje D`

Prvo se pregledaju **sve** kutije. Tek kad je poslednja pregledana, počinje farbanje. Između te dve
faze postoji **gomila** — niz od tri ispravne kutije koji je morao negde da stane.

**Jedna traka:** `pregled A → farbanje A → pregled B → pregled C → farbanje C → pregled D → farbanje D`

Kutija A prođe **ceo put** pre nego što B uopšte krene. Nema trenutka u kome negde postoji gomila
poluobrađenih kutija. Primeti i da posle `pregled B` nema farbanja — B je oštećena, pa je izbačena
i nikad nije stigla do drugog radnika.

### Šta transdjuser zapravo jeste

Pogledaj ponovo oblik:

```js
const preslikaj = posao => sledeci => (stanje, kutija) => sledeci(stanje, posao(kutija));
//                 ↑         ↑          ↑
//              podesi    kome da     ovo je opet redjuser
//              radnika   predaje
```

Tri nivoa, i svaki ima svoje ime:

1. `posao` — **podešavanje radnika**: šta on radi. Dešava se jednom, pri sastavljanju trake.
2. `sledeci` — **veza ka sledećem**: kome predaje. Isto jednom, pri sastavljanju.
3. `(stanje, kutija)` — **sam rad**: dešava se za svaku kutiju.

Poslednji red je **redjuser istog oblika** kao pakerica iz odeljka 2. Zato se radnici mogu nizati:
izlaz svakog spajanja je opet nešto što `reduce` razume.

Otud i definicija: **transdjuser je funkcija koja prima redjuser i vraća redjuser.**
Radnik koji dobije sledećeg u nizu i vrati sebe, povezanog.

---
# 4 · Kompozicija

## Slika: raspored radnika duž trake

Ručno gnežđenje `a(b(c(kraj)))` je nečitljivo čim radnika ima više od dva. Kompozicija je funkcija
koja uzme **spisak radnika po redu kojim stoje na traci** i sama ih poveže.

Jedna zamka koju vredi razumeti odmah:

- Kod **običnih** funkcija, `spoji(a, b)(x)` znači `a(b(x))` — dakle **b prvo**.
- Kod **transdjusera**, `spoji(a, b)` znači da je **a prvi na traci**.

Zvuči kontradiktorno, ali nije: kod transdjusera se ne spajaju poslovi nego **omotači oko redjusera**,
pa se obrtanje desi dvaput i poništi. Praktično pravilo: **kod transdjusera je redosled pisanja
i redosled na traci isti.**

In [ ]:
const poredjaj = (...radnici) => kraj => radnici.reduceRight((veza, r) => r(veza), kraj);

const teska = k => k.tezina > 5;

// tri radnika: izbaci oštećene → izbaci lagane → ofarbaj
const pogon = poredjaj(izdvoj(ispravna), izdvoj(teska), preslikaj(ofarbaj));

console.log("kroz tri radnika:", kutije.reduce(pogon(uGajbu), []).map(k => k.oznaka));

// ISTI raspored, druga pakerica — radnici se ne diraju
console.log("ista traka, vaga: ", kutije.reduce(pogon(izmeri), 0));
console.log("ista traka, brojac:", kutije.reduce(pogon(prebroj), 0));

// dokaz da je redosled pisanja = redosled na traci
const trag = [];
const zabelezi = ime => sledeci => (stanje, k) => { trag.push(`${ime}(${k.oznaka})`); return sledeci(stanje, k); };
kutije.reduce(poredjaj(zabelezi("prvi"), zabelezi("drugi"), zabelezi("treci"))(uGajbu), []);
console.log("\nredosled za prvu kutiju:", trag.slice(0, 3).join(" → "));

### Šta se time dobilo

`pogon` je sada **vrednost** — raspored radnika koji se može čuvati, proslediti, iskoristiti na drugom
mestu ili proširiti. Nije naredba nego opis linije.

Tri odluke koje su u običnom `niz.filter(p).map(f)` slepljene, ovde su razdvojene:

| Odluka | Ko je donosi |
| --- | --- |
| šta se radi sa svakom stavkom | radnici (transdjuseri) |
| šta je konačan ishod | pakerica (krajnji redjuser) |
| odakle stavke dolaze | `reduce` i njegov izvor |

Zbog tog razdvajanja isti radnici rade nad nizom, nad tokom podataka, nad redom poruka — bilo čime
što se može provući kroz `reduce`.

---
# 5 · Asinhrono i `Promise`

## Slika: priznanica u pekari

Naručiš burek. Ne dobijaš burek — dobijaš **papirić sa brojem**. Papirić postoji **odmah**,
burek postoji **kasnije**.

Šta možeš sa papirićem:

- **staviti ga u džep** i raditi nešto drugo (program se ne blokira),
- **unapred reći šta ćeš kad burek stigne**: „kad me prozovu, poliću ga jogurtom" — to je `.then`,
- **dati papirić nekom drugom** da preuzme umesto tebe (Promise se prosleđuje kao vrednost),
- **stati i čekati kod pulta** dok ne prozovu — to je `await`.

Šta ne možeš: pojesti papirić. `Promise` **nije** vrednost — to je obećanje da će vrednost stići.
Zato `console.log(f())` bez `await` ispiše `Promise { <pending> }`, a ne rezultat.

Još jedna bitna stvar: `async` funkcija **uvek** vraća papirić, čak i kad je posao trenutan.
Time svi pozivi izgledaju isto, pa se brzi i spori koraci mogu mešati bez razmišljanja.

In [ ]:
const pekara = {
    naruci: (sta, sekundi) => new Promise(gotovo =>
        setTimeout(() => gotovo(`${sta} (pečen za ${sekundi}s)`), sekundi * 100))
};

const papiric = pekara.naruci("burek", 3);

console.log("odmah dobijem:", papiric);
console.log("da li je to burek:", typeof papiric === "string" ? "jeste" : "nije, to je priznanica");

// unapred kažem šta ću kad stigne
papiric.then(jelo => console.log("kasnije stigne:", jelo));

console.log("u međuvremenu radim nešto drugo...");

// await = stanem kod pulta i čekam
const jelo = await papiric;
console.log("posle await-a imam:", jelo);

// async uvek vraća priznanicu, i kad nema čekanja
const trenutno = async () => 42;
console.log("\ntrenutna funkcija vraća:", trenutno());
console.log("sa await:", await trenutno());

### Zašto je to korisno

Za razliku od `Observable`-a iz Angulara, priznanica važi za **tačno jednu** isporuku i posao
kreće odmah — poređenje je u P4 na kraju.

Zato što se **plan** može napraviti pre nego što podaci stignu. `.then` je „upiši u nalog šta uraditi
kad bude gotovo", a ne „čekaj ovde". Program tako opisuje **redosled zavisnosti**, a ne redosled
u vremenu — a kad će se šta stvarno desiti, odlučuje izvršno okruženje.

---
# 6 · Sekvencijalno izvršavanje

## Slika: šalteri u opštini

Treba ti uverenje. Postupak je:

1. **Šalter 1** ti da potvrdu o prebivalištu.
2. **Šalter 2** primi *tu potvrdu* i da ti overu.
3. **Šalter 3** primi *tu overu* i izda uverenje.

Ne možeš na šalter 2 pre šaltera 1 — ne zato što je red, nego zato što **nemaš šta da mu daš**.
Izlaz jednog je ulaz sledećeg. Redosled je posledica **zavisnosti podataka**, ne pravila.

Sasvim drugi slučaj: tri nezavisne potvrde koje samo treba prikupiti. Tada možeš poslati tri osobe
na tri šaltera istovremeno. To je `Promise.all` — i **nije** ono što traži ovaj obrazac.

## Kako se to piše bez petlje

Lanac se pravi preklapanjem, gde je akumulator **priznanica koja nosi ulaz sledećeg koraka**:

```js
salteri.reduce((tok, salter) => tok.then(salter), Promise.resolve(pocetniPapir))
```

Prevedeno na srpski: *„počni sa početnim papirom u ruci; za svaki šalter dopiši u plan — kad prethodno
bude gotovo, odnesi to ovom šalteru".*

In [ ]:
const salter = (naziv, trajanje) => papir =>
    new Promise(gotovo => setTimeout(() => {
        console.log(`   ${naziv}: primio "${papir}"`);
        gotovo(`${papir} + ${naziv}`);
    }, trajanje));

const salteri = [salter("prebivaliste", 60), salter("overa", 30), salter("uverenje", 40)];

console.log("── redom, jer izlaz jednog je ulaz sledećeg ──");
const pocetak = Date.now();
const uverenje = await salteri.reduce((tok, s) => tok.then(s), Promise.resolve("zahtev"));
console.log("rezultat:", uverenje, `| trajalo: ${Date.now() - pocetak > 100 ? "zbir svih" : "kratko"}`);

console.log("\n── uporedo, kad koraci NE zavise jedan od drugog ──");
const nezavisni = [salter("A", 60), salter("B", 30), salter("C", 40)];
const svi = await Promise.all(nezavisni.map(s => s("zahtev")));
console.log("rezultat:", svi.length, "papira odjednom");

### Na šta paziti

Primeti u ispisu da kod redom-varijante svaki šalter prima **ono što je prethodni izdao**
(`zahtev`, pa `zahtev + prebivaliste`, …), dok kod `Promise.all` sva trojica primaju **isti početni**
zahtev. To je cela razlika.

Uobičajena greška je napisati `salteri.map(async s => await s(papir))` i očekivati redosled.
`map` pokrene **sve odjednom** i svakom da isti papir — dobiće se tri nezavisna posla, ne lanac.

---
# 7 · Iterabilni protokol

## Slika: redomat u banci

Na zidu stoji aparat. Pritisneš dugme — izađe jedan papirić sa brojem. Pritisneš opet — sledeći.
U jednom trenutku aparat kaže da je rolna gotova.

Da bi nešto bilo „redomat", ne mora da bude rolna papira. Mora samo da ima:

- **dugme** — mesto na koje se pritiska, uvek isto ime: `Symbol.iterator`,
- iza dugmeta nešto što na svaki pritisak vrati **sledeću vrednost** i **da li je kraj**.

Zbog toga `for…of`, `[...nesto]`, razlaganje i `Array.from` rade nad **svim** što ima to dugme —
nizom, stringom, `Map`-om, `Set`-om, i nad bilo čim što sami napravimo. Ne mora nasleđivati od niza,
ne mora ni čuvati vrednosti — mora samo znati **koja je sledeća**.

## Generator: aparat koji štampa po pozivu

Rolna može biti unapred odštampana, ali ne mora. **Generator** (`function*`) pravi vrednosti tek kad
se pritisne dugme. `yield` je „evo jednog papirića, sad stani i čekaj sledeći pritisak".

Zato generator može da opiše i **beskonačan** niz: dok se ne pritisne, ništa se ne računa.

In [ ]:
// običan objekat koji nije niz, ali JESTE iterabilan
const rolna = {
    brojevi: [101, 102, 103],
    [Symbol.iterator]() {
        let i = 0;
        const b = this.brojevi;
        return { next: () => i < b.length ? { value: b[i++], done: false } : { value: undefined, done: true } };
    }
};

console.log("for...of nad objektom:");
for (const broj of rolna) console.log("   papirić:", broj);
console.log("spread:", [...rolna], "| Array.from:", Array.from(rolna));
console.log("razlaganje:", (([prvi, drugi]) => ({ prvi, drugi }))(rolna));

// isto to, kraće — generator piše iterator umesto nas
const rolna2 = { *[Symbol.iterator]() { yield 201; yield 202; yield 203; } };
console.log("\ngenerator kao dugme:", [...rolna2]);

// aparat koji štampa po pozivu — beskonačno, a ne zaglavljuje se
function* redomat(od = 1) { let n = od; while (true) yield n++; }

const uzmi = (koliko, izvor) => {
    const r = [];
    for (const x of izvor) { if (r.length >= koliko) break; r.push(x); }
    return r;
};

console.log("prvih 5 iz beskonačnog:", uzmi(5, redomat()));
console.log("prvih 3 od 100:        ", uzmi(3, redomat(100)));

### Zašto protokol, a ne tip

Kod koji pita *„da li je ovo niz"* radi samo sa nizovima. Kod koji pita *„ima li ovo dugme"* radi sa
svime što neko ikad napravi, uključujući stvari koje nisu postojale kad je taj kod pisan.

To je razlika između provere **tipa** i provere **sposobnosti** — i razlog zašto se ugnježđene strukture
mogu izravnati bez ijedne provere kome tačno pripadaju.

---
# 8 · `map`, `filter`, `flat`, `flatMap` nad kontejnerom

## Slika: paketi u pošti

Zamisli **kutiju sa stvarima**. Četiri različite radnje:

| Radnja | Šta se dešava | Šta ostaje |
| --- | --- | --- |
| **map** | svaku stvar zameniš drugom | ista kutija, drugačiji sadržaj |
| **filter** | neke stvari izbaciš | ista kutija, manje stvari |
| **flat** | otvoriš kutije koje su unutar kutije i sve prospeš u glavnu | jedan sloj pakovanja manje |
| **flatMap** | svaku stvar zameniš **kutijicom**, pa sve kutijice odmah otvoriš | ista kutija, sve rasuto |

Bitno kod sve četiri: **dobijaš novu kutiju, stara ostaje kakva je bila.** Zato se mogu nizati
jedna za drugom, i zato uvek vraćaju isti *oblik* (kutiju), a ne golu gomilu — inače se lanac prekida.

`flatMap` postoji zato što je „napravi kutijicu za svaku stvar, pa raspakuj" toliko čest par da mu se
dalo ime. Bez njega bi to bilo `map` pa `flat`.

In [ ]:
// naš kontejner: kutija koja zna ove četiri radnje i vraća NOVU kutiju
const Kutija = (sadrzaj = []) => {
    const s = Object.freeze([...sadrzaj]);
    const jeIterabilan = x => x != null && typeof x[Symbol.iterator] === "function";
    const izravnaj = niz => niz.flatMap(x => jeIterabilan(x) ? [...x] : [x]);

    return Object.freeze({
        get stvari() { return [...s]; },
        *[Symbol.iterator]() { yield* s; },
        map:     f => Kutija(s.map(f)),
        filter:  p => Kutija(s.filter(p)),
        flat:    () => Kutija(izravnaj(s)),
        flatMap: f => Kutija(izravnaj(s.map(f)))
    });
};

const polazna = Kutija(["knjiga", "sat", "cokolada"]);

console.log("map:   ", [...polazna.map(x => x.toUpperCase())]);
console.log("filter:", [...polazna.filter(x => x.length > 3)]);

// kutija u kojoj su druge kutije
const ugnjezdena = Kutija([Kutija(["a", "b"]), Kutija(["c"]), "d"]);
console.log("\npre flat: ", ugnjezdena.stvari.length, "stvari (dve su kutije)");
console.log("posle flat:", [...ugnjezdena.flat()]);

console.log("\nflatMap — svaka stvar postaje kutijica pa se raspakuje:");
console.log("  ", [...polazna.flatMap(x => Kutija([x, x.length]))]);

console.log("\nnizanje radnji:", [...polazna.filter(x => x !== "sat").map(x => x[0])]);
console.log("polazna netaknuta:", [...polazna]);

### Zašto sve vraća kutiju

Da bi se moglo nizati. `polazna.filter(...).map(...)` radi samo ako `filter` vrati nešto što opet ima
`map`. Da vraća goli niz, lanac bi se prekinuo posle prvog koraka i morao bi se ručno prepakivati.

To je razlog zašto se kontejner ne pravi tako što nasleđuje niz, nego tako što **ponavlja isti oblik**
na izlazu svake radnje.

---
# 9 · Dekorisanje

## Slika: sekretarica ispred direktora

Direktor donosi odluke. Uprava hoće da se svaki zahtev **zabeleži** — ko je tražio, šta je odgovoreno.

Loše rešenje: naterati direktora da uz svaku odluku sam zapisuje u knjigu. Sad direktor radi dva posla,
i ako se sutra traži i merenje vremena — treći.

Dobro rešenje: **sekretarica ispred vrata**. Ona primi zahtev, zapiše ga, prosledi direktoru, sačeka
odgovor, zapiše i njega, i vrati odgovor. Za onoga ko dolazi ništa se nije promenilo — dobija isti
odgovor kao i pre. Direktor se **uopšte nije menjao**.

Tri osobine koje to čini upotrebljivim:

1. **Isti ugovor.** Sekretarica prima isto što bi primio direktor i vraća isto što bi on vratio.
   Zato se može staviti bilo gde gde je stajao direktor.
2. **Original netaknut.** Direktor i dalje postoji sam za sebe i može se koristiti bez sekretarice.
3. **Slaganje.** Ispred sekretarice može stati još neko — recimo onaj ko meri koliko je čekanje trajalo.

To je sve što „dekorisanje" jeste: **novi sloj sa istim ugovorom, koji oko originala dopisuje posao.**

In [ ]:
const direktor = zahtev => `odobreno: ${zahtev}`;

// sekretarica: prima funkciju, vraća funkciju ISTOG oblika
const saBelezenjem = (ime, fja) => (...args) => {
    console.log(`   → ${ime} prima:`, ...args);
    const odgovor = fja(...args);
    console.log(`   ← ${ime} vraća:`, odgovor);
    return odgovor;
};

const saMerenjem = (ime, fja) => (...args) => {
    const t = Date.now();
    const odgovor = fja(...args);
    console.log(`   ⏱ ${ime}: ${Date.now() - t} ms`);
    return odgovor;
};

console.log("bez sloja:", direktor("odmor"));

console.log("\njedan sloj:");
const saSekretaricom = saBelezenjem("direktor", direktor);
console.log("rezultat:", saSekretaricom("odmor"));

console.log("\ndva sloja, jedan preko drugog:");
const dvaSloja = saMerenjem("ukupno", saBelezenjem("direktor", direktor));
console.log("rezultat:", dvaSloja("povisica"));

console.log("\noriginal nije ni pipnut:", direktor("odmor"));
console.log("isti objekat:", saSekretaricom !== direktor);

### Veza sa odeljkom 3

Sekretarica je **isti obrazac** kao radnik pored trake: dobije nekoga kome prosleđuje, i vrati sebe
sa tom vezom. Razlika je samo u tome šta prolazi kroz nju — kroz radnika prolaze kutije, kroz
sekretaricu prolaze pozivi.

Zato se dekorisanje i transdjuseri tako lako spajaju: ako se stavke koje idu trakom i same
mogu izvršavati, onda je „radnik koji svaku stavku zameni njenom dekorisanom verzijom"
sasvim običan `preslikaj`.

---
# 10 · Nepromenljivost

## Slika: spisak zakucan na tabli

Na oglasnoj tabli visi spisak dežurnih. Dva načina da se doda ime:

- **Precrtaš i dopišeš** na postojećem papiru. Svako ko je papir već fotografisao ima pogrešnu sliku,
  a stara verzija više ne postoji.
- **Odštampaš nov spisak** i zakucaš ga pored, sa strelicom „prethodni". Ko gleda stari — vidi tačno
  ono što je i pre video. Istorija postoji sama od sebe, jer nijedan papir nije uništen.

Drugi način je nepromenljivost. Deluje rasipnički, ali papiri **dele sadržaj**: nova verzija ne
prepisuje sva imena, samo pokazuje na iste zapise plus jedan nov.

In [ ]:
const Spisak = (imena = [], prethodni = null) => {
    const s = Object.freeze([...imena]);
    const ovaj = Object.freeze({
        prethodni,
        get imena() { return [...s]; },
        dodaj: ime => Spisak([...s, ime], ovaj),
        ukloni: ime => Spisak(s.filter(x => x !== ime), ovaj)
    });
    return ovaj;
};

const v0 = Spisak();
const v1 = v0.dodaj("Ana");
const v2 = v1.dodaj("Marko");
const v3 = v2.ukloni("Ana");

console.log("v1:", v1.imena, "| v2:", v2.imena, "| v3:", v3.imena);
console.log("stare verzije netaknute:", v0.imena, v1.imena);

const lanac = s => s === null ? [] : [...lanac(s.prethodni), s];
console.log("istorija:", lanac(v3).map(s => s.imena));

// geter vraća kopiju — spolja se ne može upisati
v2.imena.push("Iva");
console.log("posle pokušaja upisa:", v2.imena);

---
# Kako se sve spaja

Deset odeljaka nisu deset nezavisnih tema — to je jedan lanac:

1. **Funkcija je podatak** (1) → zato se posao može proslediti spolja.
2. Ako se posao prosleđuje, treba oblik koji ga prima. Za skupljanje to je **redjuser** (2).
3. Ako se hoće više poslova nad istim tokom bez međugomila, radnik mora znati **kome predaje** —
   tako nastaje **transdjuser** (3), a **kompozicija** (4) ih ređa.
4. Kad posao traje, rezultat je **priznanica** (5), a kad izlaz jednog hrani sledećeg, lanac se pravi
   **preklapanjem po priznanicama** (6) — isti `reduce` kao u odeljku 2, samo je akumulator obećanje.
5. Da bi bilo šta uopšte teklo, izvor mora umeti da da **sledeću stavku** (7).
6. Ako se te stavke drže u nekom kontejneru, on nudi **map/filter/flat/flatMap** (8) i uvek vraća
   sebi sličan kontejner.
7. Kad treba dodati sporedni posao — ispis, merenje, proveru — dodaje se **sloj oko** (9), a ne izmena
   unutra.
8. A sve to je bezbedno jer se ništa ne prepravlja, nego se pravi **nova verzija** (10).

## Tri rečenice koje vrede više od definicija

> **Redjuser** kaže *šta je ishod*. **Transdjuser** kaže *šta se usput radi*. **Izvor** kaže *odakle stavke*.
> Tri odluke, tri mesta — i nijedno ne zna za drugo.

> `Promise` nije vrednost, nego **priznanica** da će vrednost stići. Zato se plan pravi pre nego što
> podaci postoje.

> Protokol je **sposobnost**, ne tip. Pitanje nije „šta je ovo", nego „ume li ovo da mi da sledeću stvar".

---
# Pitanja

## P1 · „Onaj ko ga izvodi ostaje nepromenjen" — ko je to tačno

Izvršilac, ne pozivalac. Reč je o **funkciji koja prima recept i primenjuje ga** — u primeru iz
odeljka 1 to je `posluziTrojici`. Pozivalac se, naprotiv, menja stalno: svaki put šalje drugi recept.

| Uloga | Ko je to | Menja li se |
| --- | --- | --- |
| **pozivalac** | kod koji piše `posluzi(recept)` | da — bira recept |
| **izvršilac** | `posluzi` — obilazi i primenjuje | **ne** |
| **posao** | recept | prosleđuje se kao vrednost |

## P2 · Razlika između redjusera i `reduce`

**Redjuser** je funkcija koju **ti pišeš** — jedan korak: `(stanje, stavka) => novo stanje`.
Ne zna da postoji niz, ne zna koliko ima stavki, ne zna gde je početak ni kraj.

**`reduce`** je **mašina** koja obilazi izvor i taj jedan korak ponavlja, noseći akumulator kroz obilazak.

U slici sa kasom: redjuser je **kasirka** — šta radi sa jednim artiklom. `reduce` je **traka plus
pravilo** „propusti sve artikle kroz nju i reci šta je na kraju na displeju".

`reduce` nije ništa magično; u ćeliji ispod je napisan ručno u pet redova. Poenta je poslednji deo
ispisa: **isti redjuser radi nad nizom, nad generatorom i nad `Set`-om**, jer ne zna odakle stavke
dolaze. To zna `reduce`. Ta podela je razlog zašto se radnici sa trake mogu preseliti u bilo koji pogon.

In [ ]:
// ── P1 · izvršilac se ne menja, poziv se menja ───────────────────────

const posluziA = () => ["Ana", "Marko"].map(ime => `Zdravo, ${ime}!`);   // posao UGRAĐEN
const posluziB = recept => ["Ana", "Marko"].map(recept);                 // posao STIŽE spolja

console.log("A, jedini mogući ishod:", posluziA());
console.log("B, prvi recept: ", posluziB(i => `Zdravo, ${i}!`));
console.log("B, drugi recept:", posluziB(i => i.length));
console.log("B, treći recept:", posluziB(i => i.toUpperCase()));
console.log("→ posluziB nije menjan nijednom; menjao se samo POZIV");

// ── P2 · reduce je samo mašina; redjuser je korak ────────────────────

const mojReduce = (izvor, redjuser, pocetno) => {
    let stanje = pocetno;
    for (const stavka of izvor) stanje = redjuser(stanje, stavka);
    return stanje;
};

const qSaberi = (zbir, x) => zbir + x;          // REDJUSER — jedan korak
const qBrojevi = [1, 2, 3, 4];

console.log("\nugrađeni reduce:", qBrojevi.reduce(qSaberi, 0));
console.log("moj reduce:     ", mojReduce(qBrojevi, qSaberi, 0), "— isti redjuser, druga mašina");

// isti redjuser nad izvorima koji NISU niz
function* qTok(od, koliko) { for (let i = 0; i < koliko; i++) yield od + i; }

console.log("isti redjuser nad generatorom:", mojReduce(qTok(10, 4), qSaberi, 0));
console.log("isti redjuser nad Set-om:     ", mojReduce(new Set([5, 5, 7]), qSaberi, 0));
console.log("→ redjuser ne zna odakle stavke dolaze; reduce zna");

## P3 · Zašto tri nivoa, a ne dva ili jedan

```js
const triNivoa = posao => sledeci => (stanje, kutija) => sledeci(stanje, posao(kutija));
const dvaNivoa = (posao, sledeci) => (stanje, kutija) => sledeci(stanje, posao(kutija));
const ravno    = (posao, sledeci, stanje, kutija) => sledeci(stanje, posao(kutija));
```

Sva tri **računaju istu stvar**. Razlika je u tome **kad smeš da daš koji argument** — i, posledično,
šta možeš da držiš kao vrednost između.

**Ravna verzija ne može da se preda `reduce`-u.** `reduce` zove redjuser sa `(stanje, stavka)`, pa bi
akumulator upao u `posao` i dobio bi se `TypeError: posao is not a function`. Radi tek uz ručno
umotavanje `(s, k) => ravno(ofarbaj, uGajbu, s, k)` — **a to umotavanje je upravo kariranje**, samo
napisano rukom na svakom mestu upotrebe.

**Verzija sa dva nivoa** jeste ispravan redjuser i radi sa `reduce`. Puca na kompoziciji.
Pogledaj `poredjaj` iz odeljka 4:

```js
const poredjaj = (...radnici) => kraj => radnici.reduceRight((veza, r) => r(veza), kraj);
//                                                            ↑ radniku se daje TAČNO JEDAN argument
```

`poredjaj` radniku daje **samo sledećeg u nizu**. Nema odakle da mu doda `posao` — njega je trebalo
zadati ranije, pri podešavanju. Sa dva nivoa ne postoji trenutak u kome imaš „podešenog, ali još
nepovezanog radnika", a to je tačno ono što kompozicija prima.

**Tri nivoa = tri različita trenutka:**

| Nivo | Kad se dešava | Koliko puta |
| --- | --- | --- |
| `posao` | podešavanje radnika | jednom |
| `sledeci` | sastavljanje trake | jednom |
| `(stanje, kutija)` | sam rad | po kutiji |

I to nije samo teorija — ćelija ispod broji koliko se puta spoljni deo stvarno izvrši.

**Pravilo:** koliko različitih trenutaka, toliko strelica.

In [ ]:
// koristimo radnike i kutije iz odeljka 3 i 4
const qDvaNivoa = (posao, sledeci) => (stanje, kutija) => sledeci(stanje, posao(kutija));
const qRavno    = (posao, sledeci, stanje, kutija) => sledeci(stanje, posao(kutija));

console.log("tri nivoa (preslikaj):", kutije.reduce(preslikaj(ofarbaj)(uGajbu), []).map(k => k.boja));
console.log("dva nivoa:            ", kutije.reduce(qDvaNivoa(ofarbaj, uGajbu), []).map(k => k.boja));

try {
    kutije.reduce(qRavno, []);
} catch (g) {
    console.log("ravno direktno →", g.constructor.name + ":", g.message.slice(0, 40));
}
console.log("ravno uz umotavanje:  ",
            kutije.reduce((s, k) => qRavno(ofarbaj, uGajbu, s, k), []).map(k => k.boja),
            "— a to umotavanje JE kariranje");

// kompozicija radi samo sa tri nivoa
const qTraka = poredjaj(izdvoj(ispravna), preslikaj(ofarbaj));
console.log("\nporedjaj sa tri nivoa:", kutije.reduce(qTraka(uGajbu), []).map(k => k.oznaka));
console.log("sa dva nivoa nemoguće: poredjaj radniku daje SAMO sledećeg,");
console.log("   a qDvaNivoa traži (posao, sledeci) zajedno — nema gde da primi posao");

// da li se spoljni deo računa po kutiji ili jednom?
let qBrojac = 0;
const qSaBrojanjem = posao => { qBrojac++; return sledeci => (s, k) => sledeci(s, posao(k)); };
kutije.reduce(qSaBrojanjem(ofarbaj)(uGajbu), []);
console.log("\npodešavanje radnika izvršeno puta:", qBrojac, "za", kutije.length, "kutije");

## P4 · `.then` naspram `.subscribe` u Angularu

Najkraće: `Promise` je **priznanica za jednu isporuku**, `Observable` je **pretplata na isporuke**.

Pravu sliku daje podela po dve ose:

| | **jedna vrednost** | **više vrednosti** |
| --- | --- | --- |
| **pull** — ti tražiš | obična funkcija | `Iterable` / generator *(odeljak 7)* |
| **push** — tebi javljaju | **`Promise`** *(odeljak 5)* | **`Observable`** |

`Observable` je tačno ono što nedostaje u tom kvadratu — tok koji **gura** više vrednosti kroz vreme.
Iterator čeka da pritisneš dugme; tok te sam zove kad nešto stigne.

| | `Promise` / `.then` | `Observable` / `.subscribe` |
| --- | --- | --- |
| koliko vrednosti | tačno jedna | nula, jedna, više, beskonačno |
| kad kreće | odmah pri nastanku | tek na pretplatu (hladni tokovi) |
| ponavljanje | ne — rezultat keširan | svaka pretplata pokreće iznova |
| otkazivanje | nema | `unsubscribe()` |
| obrada usput | lanac `.then` | `pipe(map(…), filter(…))` |

### Veza sa odeljkom 3

`pipe(map(f), filter(p))` **je traka sa kutijama**: operatori su radnici, `subscribe` je pakerica na
kraju. Kao i kod transdjusera, `pipe` **ne pravi međunizove** — svaka vrednost prođe kroz sve operatore
pre nego što sledeća krene, tačno kao u ispisu iz odeljka 3.

Nisu doslovno isti mehanizam — RxJS operator je `Observable => Observable`, a transdjuser
`redjuser => redjuser` — ali je ideja identična: sloj koji prima sledeći sloj i vraća sebe povezanog.

### U Angularu, praktično

- `HttpClient` vraća **hladan** `Observable` — zahtev kreće tek na `subscribe`, i **svaka pretplata
  šalje nov zahtev**. Klasična greška je pretplatiti se dvaput i čuditi se dvama pozivima ka serveru.
- `async` pipe se pretplaćuje i **otpretplaćuje sam** — otud preporuka umesto ručnog `subscribe`.
- `firstValueFrom(obs)` pretvara tok u `Promise` kad ti stvarno treba samo prva vrednost.
- Tamo gde vrednosti ima više — `valueChanges` na formi, parametri rute, WebSocket — `Promise` ne bi
  ni mogao da posluži, jer isporučuje tačno jednom.

Ćelija ispod pravi minimalni tok u petnaest redova, da se razlike vide bez Angulara.

In [ ]:
// ── Promise: kreće ODMAH i pamti rezultat ────────────────────────────
let qPokrenuto = 0;
const qP = new Promise(gotovo => { qPokrenuto++; setTimeout(() => gotovo("vrednost"), 10); });

console.log("posao pokrenut pre ijednog .then:", qPokrenuto);
qP.then(v => console.log("   prvi .then:", v));
qP.then(v => console.log("   drugi .then:", v, "— posao NIJE ponovljen, rezultat je keširan"));
await qP;

// ── minimalni tok: javlja VIŠE puta, kreće tek na pretplatu, otkaziv ──
const Tok = proizvodjac => ({
    subscribe: posmatrac => {
        let ziv = true;
        proizvodjac({ next: v => ziv && posmatrac.next(v), done: () => ziv && posmatrac.done?.() });
        return { otkazi: () => { ziv = false; } };
    }
});

let qPokrenutoTok = 0;
const qTok2 = Tok(({ next, done }) => { qPokrenutoTok++; [1, 2, 3].forEach(next); done(); });

console.log("\ntok napravljen, posao pokrenut puta:", qPokrenutoTok, "— ništa dok se ne pretplatiš");

qTok2.subscribe({ next: v => console.log("   pretplatnik 1:", v),
                  done: () => console.log("   pretplatnik 1: kraj") });
qTok2.subscribe({ next: v => console.log("   pretplatnik 2:", v) });

console.log("posao pokrenut puta:", qPokrenutoTok, "— jednom PO pretplati, ne kešira se");
console.log("otkazivanje postoji kod toka, kod Promise-a ne");